In [1]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, InsulationMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors import TablessCurrentCollector
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups import Laminate

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [2]:
db = DataManager()

In [3]:
db.remove_data(table_name='cells', condition="name = 'QNAS-S40160NL'")

In [4]:
db.get_table_names()

['cells',
 'anode_materials',
 'binder_materials',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'separator_materials',
 'cathode_materials']

In [5]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation_material = InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = SeparatorMaterial.from_database("Polypropylene")


In [6]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=151,
    length=2975,
    coated_width=142,
    insulation_width=3,
    thickness=13
)

cathode_active_material = CathodeMaterial.from_database("NFM111 (Vendor C)")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=3.09,
    mass_loading=21.70,
    insulation_material=insulation_material,
    insulation_thickness=3
)



In [7]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=153,
    length=2980,
    coated_width=144,
    insulation_width=3,
    thickness=13
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.05,
    mass_loading=8,
    insulation_material=insulation_material,
    insulation_thickness=3
)

In [8]:
# make the layup

top_separator = Separator(
    material=separator_material,
    width=286,
    length=4900,
    thickness=25
)

bottom_separator = Separator(
    material=separator_material,
    width=286,
    length=4900,
    thickness=25
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_separator,
    name="QNAS-S40160NL",
)


In [9]:
my_cathode.properties

{'Cost': '$ 2.19',
 'Mass': '195.43 g',
 'Coating mass': '179.47 g',
 'Total thickness': '153.5 um',
 'Coating thickness': '70.23 um'}

In [10]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_layup.name],
    'object': [my_layup.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [11]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,QNAS-S40160RL,gASVbwMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-09-19 13:24:28,0.3.10,Na/Na+
1,UniGrid-NCO32140,gASVb64AAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-09-19 13:24:47,0.3.10,Na/Na+
2,Vendor D NFPP,gASVY+cAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-09-19 13:25:09,0.3.10,Na/Na+
3,Vendor E NFPP,gASVY+cAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-09-19 13:25:27,0.3.10,Na/Na+
4,Vendor G NFM,gASVbwMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-09-19 13:26:01,0.3.10,Na/Na+
5,Vendor G NFPP,gASVCuAAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-09-19 13:26:19,0.3.10,Na/Na+
6,CBAK-32140NS,gASVbwMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-09-22 17:03:12,0.3.10,Na/Na+
7,Cell 2,gASVagkBAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-09-22 17:03:14,0.3.10,Na/Na+
8,Cell 4,gASVe/8AAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-09-22 17:03:16,0.3.10,Na/Na+
9,HiNa-NaCR32140-MP10,gASVbwMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-09-22 17:03:18,0.3.10,Na/Na+
